# カオス 1：ロジスティック写像 ―決定論なのに予測できない

[学習ロードマップ](../../../docs/learning_roadmap.md)の候補14（カオスと力学系）を、**ノートブックから**始めます。式の導出は後でノートにまとめるので、ここではまず計算して、結果を見て考えます。各節の最後に「**考えてみよう**」という問いを置きました。

扱うのは、いちばん単純なカオスの例である**ロジスティック写像**

$$
x_{n+1}=r\,x_n(1-x_n)\qquad(0\le x_n\le1,\ 0\le r\le4)
$$

です。$x_n$ は「生き物の数（最大を 1 とした割合）」、$r$ は増え方の強さ、という解釈がよく使われます。式は1行で、乱数も使っていません（**決定論的**）。それでも、$r$ によっては先が読めなくなります。

**中心になる量**は**リアプノフ指数** $\lambda$ です。近い2つの初期値の差が $|\delta_n|\approx|\delta_0|\,e^{\lambda n}$ と広がる速さで、$\lambda>0$ がカオスの目印になります。

**使い方**：上から順に実行します。`assert` は、計算結果が既知の値と合っていることを確かめるものです。

In [ ]:
import numpy as np
import plotly.graph_objects as go
import ipywidgets as W
from numpy.typing import NDArray
from plotly.subplots import make_subplots

Floats = NDArray[np.float64]   # 実数の配列（型ヒント用の別名）
# 注意：FigureWidget（スライダーで描き直す図）には、numpy 配列ではなくリストを渡す（NB-02 を参照）


def logistic(r: float | Floats, x: float | Floats) -> float | Floats:
    """ロジスティック写像 f(x) = r x (1 − x)"""
    return r * x * (1 - x)


def orbit(r: float, x0: float, n: int) -> Floats:
    """x0, x1, …, x_{n−1}（初期値 x0 から n 個）"""
    xs = np.empty(n)
    xs[0] = x0
    for k in range(1, n):
        xs[k] = logistic(r, xs[k - 1])
    return xs

## 1. $r$ を変えると、振る舞いが変わる

4つの $r$ で、$x_0=0.2$ から 60 回くり返した時系列を描きます。

In [ ]:
cases = [(2.8, "r = 2.8"), (3.2, "r = 3.2"), (3.5, "r = 3.5"), (3.9, "r = 3.9")]
fig = make_subplots(rows=2, cols=2, subplot_titles=[t for _, t in cases], shared_yaxes=True)
for k, (r, title) in enumerate(cases):
    xs = orbit(r, 0.2, 60)
    fig.add_trace(go.Scatter(y=xs.tolist(), mode="lines+markers", marker=dict(size=4), showlegend=False),
                  row=k // 2 + 1, col=k % 2 + 1)
fig.update_layout(height=500, width=900, margin=dict(t=40, l=40, r=20, b=30))
fig.update_xaxes(title_text="n")

# 十分くり返した後の最後の 64 回で、何種類の値が現れるか（周期）を数える
for r, title in cases:
    tail = np.round(orbit(r, 0.2, 2000)[-64:], 6)
    print(f"{title}：くり返す値の種類 = {len(np.unique(tail))}")
fig

**考えてみよう**

- $r=2.8$ では1つの値に落ち着き、$3.2$ では2つの値、$3.5$ では4つの値を交互にくり返します。$3.9$ ではどうでしょうか。
- 1つの値に落ち着くとき、その値は $x=r x(1-x)$ の解（**不動点**）のはずです。$r=2.8$ で計算して、上の図と比べてみましょう。

## 2. 初期値への敏感さ：差は指数関数的に広がる

$r=4$ で、$10^{-10}$ だけ違う2つの初期値から始めます。差 $|x_n-y_n|$ を対数目盛りで描くと、直線（指数関数的な増加）になり、その傾きがリアプノフ指数です。$r=4$ では $\lambda=\ln2$（1回ごとに差が約2倍）であることが知られています。

In [ ]:
r, x0, d0, N = 4.0, 0.3, 1e-10, 60
xa, xb = orbit(r, x0, N), orbit(r, x0 + d0, N)
diff = np.abs(xa - xb)
n = np.arange(N)

# 差が 1e-3 以下の範囲で、log|δ_n| の傾きを最小二乗で求める
mask = diff < 1e-3
slope = np.polyfit(n[mask], np.log(diff[mask]), 1)[0]
print(f"傾き（リアプノフ指数の推定）= {slope:.3f}、ln 2 = {np.log(2):.3f}")
assert abs(slope - np.log(2)) < 0.15

fig = go.Figure([go.Scatter(x=n.tolist(), y=diff.tolist(), mode="lines+markers", name="|x_n − y_n|（計算）"),
                 go.Scatter(x=n.tolist(), y=(d0 * np.exp(np.log(2) * n)).tolist(), mode="lines",
                            line=dict(dash="dash"), name="10⁻¹⁰ · 2ⁿ（λ = ln 2）")])
fig.update_layout(height=380, width=800, yaxis_type="log", xaxis_title="n", yaxis_title="差",
                  margin=dict(t=20), yaxis=dict(range=[-11, 0.5]))
fig

**考えてみよう**

- 差は約 35 回で、$x$ の範囲（$0\sim1$）と同じ大きさまで広がり、そこで頭打ちになります。なぜ頭打ちになるのでしょうか。
- 初期値の違いを $10^{-10}$ から $10^{-15}$ に減らすと、頭打ちになるまでの回数は何回延びるでしょうか（§4 で確かめます）。
- コンピュータの数（倍精度）は、約 $10^{-16}$ の丸め誤差を含みます。$r=4$ で 60 回くり返した $x_{60}$ は、どこまで信用できるでしょうか。

## 3. リアプノフ指数と分岐図

差が1回で何倍になるかは、写像の傾き $|f'(x_n)|=|r(1-2x_n)|$ です。$n$ 回の積の対数を平均すると、リアプノフ指数

$$
\lambda=\lim_{N\to\infty}\frac1N\sum_{n=0}^{N-1}\ln|r(1-2x_n)|
$$

になります（微分は「最良の線形近似」なので、小さな差は傾き倍される）。$r$ を細かく変えて、上に**分岐図**（十分くり返した後の $x_n$ の値）、下にリアプノフ指数を描きます。

In [ ]:
def lyapunov(rs: Floats, x0: float = 0.3, transient: int = 1000, n: int = 3000) -> Floats:
    """r の配列それぞれについて、λ = (1/N) Σ ln|r(1−2x_n)| をまとめて計算する"""
    x = np.full_like(rs, x0)
    for _ in range(transient):            # 最初の変化を捨てる
        x = logistic(rs, x)
    total = np.zeros_like(rs)
    for _ in range(n):
        total += np.log(np.abs(rs * (1 - 2 * x)) + 1e-300)
        x = logistic(rs, x)
    return total / n


def bifurcation(rs: Floats, keep: int = 120, transient: int = 1000) -> tuple[Floats, Floats]:
    x = np.full_like(rs, 0.3)
    for _ in range(transient):
        x = logistic(rs, x)
    R, X = [], []
    for _ in range(keep):
        x = logistic(rs, x)
        R.append(rs.copy())
        X.append(x.copy())
    return np.concatenate(R), np.concatenate(X)


rs = np.linspace(2.5, 4.0, 1200)
lam = lyapunov(rs)
R, X = bifurcation(rs)

lam4 = lyapunov(np.array([4.0]), n=200000)[0]
print(f"r = 4 のリアプノフ指数 = {lam4:.4f}（ln 2 = {np.log(2):.4f}）")
assert abs(lam4 - np.log(2)) < 1e-2
for r_, sign in [(3.2, -1), (3.5, -1), (3.7, +1), (3.83, -1), (3.9, +1)]:
    v = lyapunov(np.array([r_]))[0]
    print(f"r = {r_}：λ = {v:+.3f}")
    assert np.sign(v) == sign

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.6, 0.4], vertical_spacing=0.05)
fig.add_trace(go.Scattergl(x=R.tolist(), y=X.tolist(), mode="markers", marker=dict(size=1, color="#1f77b4"),
                           name="分岐図"), row=1, col=1)
fig.add_trace(go.Scatter(x=rs.tolist(), y=lam.tolist(), mode="lines", line=dict(color="#d62728", width=1),
                         name="リアプノフ指数 λ"), row=2, col=1)
fig.add_hline(y=0, line_color="#555", row=2, col=1)
fig.update_yaxes(title_text="十分くり返した後の x", row=1, col=1)
fig.update_yaxes(title_text="λ", range=[-2, 1], row=2, col=1)
fig.update_xaxes(title_text="r", row=2, col=1)
fig.update_layout(height=650, width=900, margin=dict(t=20, l=60, r=20, b=40), showlegend=False)
fig

**考えてみよう**

- 分岐図で、1本の線が2本、4本、8本…と分かれていく所（**周期倍分岐**）で、$\lambda$ はちょうど $0$ に触れています。なぜでしょうか（ヒント：安定な周期軌道の上では、差は縮む。安定かどうかの境目では？）。
- $r\approx3.83$ には、カオスの中に急に3本だけになる「窓」があり、$\lambda<0$ です。カオスの中に、規則的な領域が埋まっています。
- 分岐が起きる $r$ の間隔は、だんだん縮まります。その比は $4.669\ldots$（**ファイゲンバウム定数**）に近づき、この値は写像の細かい形によらないことが知られています（このノートブックでは扱いません）。

## 4. スライダー：蜘蛛の巣図で、くり返しを見る

**蜘蛛の巣図**は、くり返しを図で追う方法です。$x_n$ から縦に放物線 $y=f(x)$ まで進み、横に対角線 $y=x$ まで進むと、そこが $x_{n+1}$ です。$r$ のスライダーを動かして、落ち着く・振動する・カオスになる様子を見ます。

In [ ]:
def cobweb(r: float, x0: float, n: int) -> tuple[list[float], list[float]]:
    xs, ys = [x0], [0.0]
    x = x0
    for _ in range(n):
        y = float(logistic(r, x))
        xs += [x, y]
        ys += [y, y]
        x = y
    return xs, ys


fig_web = go.FigureWidget(make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55],
                                    subplot_titles=("蜘蛛の巣図", "時系列 x_n")))
grid = np.linspace(0, 1, 200)
fig_web.add_trace(go.Scatter(x=grid.tolist(), y=grid.tolist(), mode="lines", line=dict(color="#999", dash="dot"), name="y = x"), 1, 1)
fig_web.add_trace(go.Scatter(x=[], y=[], mode="lines", line=dict(color="#1f77b4"), name="y = r x (1−x)"), 1, 1)
fig_web.add_trace(go.Scatter(x=[], y=[], mode="lines", line=dict(color="#d62728", width=1), name="くり返し"), 1, 1)
fig_web.add_trace(go.Scatter(x=[], y=[], mode="lines+markers", marker=dict(size=4), line=dict(color="#d62728"), name="x_n"), 1, 2)
fig_web.update_layout(height=430, width=950, margin=dict(t=40, l=40, r=20, b=40), showlegend=False,
                  xaxis=dict(range=[0, 1]), yaxis=dict(range=[0, 1], scaleanchor="x"), yaxis2=dict(range=[0, 1]))
r_slider = W.FloatSlider(value=3.2, min=2.5, max=4.0, step=0.01, description="r")
x0_slider = W.FloatSlider(value=0.2, min=0.01, max=0.99, step=0.01, description="x₀")
info = W.HTML()


def update(*_: object) -> None:
    """スライダーが動くたびに呼ばれ、蜘蛛の巣図・時系列・λ を描き直す"""
    r, x0 = r_slider.value, x0_slider.value
    cx, cy = cobweb(r, x0, 80)
    xs = orbit(r, x0, 100)
    lam_r = float(lyapunov(np.array([r]))[0])
    with fig_web.batch_update():
        fig_web.data[1].x, fig_web.data[1].y = grid.tolist(), logistic(r, grid).tolist()
        fig_web.data[2].x, fig_web.data[2].y = cx, cy
        fig_web.data[3].x, fig_web.data[3].y = list(range(100)), xs.tolist()
    kind = "カオス（λ > 0）" if lam_r > 1e-3 else ("周期倍分岐の付近（λ ≈ 0）" if lam_r > -1e-2 else "周期的（λ < 0）")
    info.value = f"r = {r:.2f}：リアプノフ指数 λ = {lam_r:+.3f} → {kind}"


for w in (r_slider, x0_slider):
    w.observe(update, "value")
update()
W.VBox([W.HBox([r_slider, x0_slider]), info, fig_web])

**考えてみよう**

- $r<3$ では、蜘蛛の巣は放物線と対角線の交点（不動点）に吸い込まれます。$r=3$ を超えると、交点のまわりを回り続けるようになります。交点での放物線の傾き $f'(x^*)$ に注目すると、境目は $|f'(x^*)|=1$ のはずです。確かめてみましょう。
- $r=3.83$ 付近で、周期3の動きを探してみましょう。

### 補足：落ち着く速さを決めるのは、$x_0$ より $r$（不動点での傾き）

蜘蛛の巣図を動かすと、$x_0$ によって落ち着きにくく見えることがあります。$x_0$ を変えて、不動点 $x^*=1-\frac1r$ にほぼ落ち着く（ずれが $10^{-6}$ 以下になる）までの回数を数えてみます。

In [ ]:
def steps_to_settle(r: float, x0: float, tol: float = 1e-6, max_n: int = 100000) -> int:
    x_star = 1 - 1 / r
    x = x0
    for n in range(max_n):
        if abs(x - x_star) < tol:
            return n
        x = logistic(r, x)
    return max_n


x0s = [0.05, 0.2, 0.5, 0.8, 0.99]
print("  r    |f'(x*)| = |2−r|   " + "  ".join(f"x0={x0:<4}" for x0 in x0s))
for r in (2.8, 2.95, 2.99):
    counts = [steps_to_settle(r, x0) for x0 in x0s]
    print(f"{r:5.2f}        {abs(2 - r):.2f}          " + "  ".join(f"{c:7d}" for c in counts))
    assert max(counts) < 1.3 * min(counts)          # x0 による違いは小さい
assert steps_to_settle(2.99, 0.2) > 15 * steps_to_settle(2.8, 0.2)   # r が 3 に近いと、桁違いに遅い

- **$x_0$ による違いはわずか**です。$r=2.8$ なら、どこから始めても約 50 回で同じ値に落ち着きます。
- **効くのは $r$ が 3 に近いかどうか**です。不動点からのずれは、1回ごとに不動点での傾き $|f'(x^*)|=|2-r|$ 倍になります。$r=2.8$ なら 0.8 倍ずつ縮みますが、$r=2.99$ では 0.99 倍ずつなので、なかなか縮みません。
- $r=3$ で倍率がちょうど 1 になり、それを超えるとずれが縮まなくなって、周期2に移ります。これが §3 の「周期倍分岐で $\lambda$ がちょうど 0 に触れる」理由です（$\lambda$ は倍率の対数の平均で、倍率 1 なら $\ln1=0$）。
- $x_0=0.99$ のように端に近い値から始めると、最初の1回で $x_1=r\cdot0.99\cdot0.01\approx0.03$ と 0 の近くへ飛び、戻るまでに数回かかります。蜘蛛の巣図で「落ち着きにくい」と見えるのは、この最初の遠回りです。

## 5. 予測できる時間

差が $\delta_0$ から許せる大きさ $\Delta$ になるまでの回数は、$\delta_0e^{\lambda n}=\Delta$ から

$$
n^*\approx\frac1\lambda\ln\frac{\Delta}{\delta_0}
$$

です。初期の精度を1桁（10倍）よくしても、予測できる回数は $\frac{\ln10}{\lambda}$（$r=4$ なら約 $3.3$ 回）しか延びません。ランダムに選んだ 300 個の初期値で平均して確かめます。

In [ ]:
def steps_to_diverge(r: float, d0: float, tol: float = 0.1, trials: int = 300, max_n: int = 200) -> float:
    rng = np.random.default_rng(0)
    x = rng.uniform(0.05, 0.95, trials)
    y = x + d0
    steps = np.full(trials, max_n, dtype=float)
    alive = np.ones(trials, dtype=bool)
    for k in range(max_n):
        x, y = logistic(r, x), logistic(r, y)
        hit = alive & (np.abs(x - y) > tol)
        steps[hit] = k + 1
        alive &= ~hit
    return float(steps.mean())


digits = np.arange(2, 15)
n_star = np.array([steps_to_diverge(4.0, 10.0 ** (-d)) for d in digits])
slope = np.polyfit(digits, n_star, 1)[0]
print(f"初期の精度 1 桁あたりの延び = {slope:.2f} 回（理論 ln10 / ln2 = {np.log(10) / np.log(2):.2f} 回）")
assert abs(slope - np.log(10) / np.log(2)) < 0.4

fig = go.Figure([go.Scatter(x=digits.tolist(), y=n_star.tolist(), mode="lines+markers", name="計算（300 個の平均）")])
fig.update_layout(height=360, width=700, xaxis_title="初期の精度（小数点以下の桁数、δ₀ = 10^−桁）",
                  yaxis_title="差が 0.1 を超えるまでの回数", margin=dict(t=20))
fig

## 観察のポイント（まとめ）

- カオスは「発散する」のではありません。$r=3.9$ でも $x$ は $0.1\sim0.97$ くらいの範囲に収まっています。特徴は次の3つがそろうことです：(1) 範囲は限られている、(2) 1つの値にも周期にも落ち着かない、(3) 近い2点の**差**が指数関数的に広がる（頭打ちになるまで）。
- 1行の決定論的な式でも、$r$ によっては (3) が起き、先が読めなくなります。差が広がる速さがリアプノフ指数 $\lambda$ で、「1回で差が何倍になるか（傾き）」の対数の平均です。倍率が 1 より小さければ落ち着き（$\lambda<0$）、1 より大きい状態が続けばカオス（$\lambda>0$）です。
- 予測できる時間は、初期の精度の**対数**にしか比例しません。精度を上げても、予測できる時間はほとんど延びません。「全てを知る悪魔なら未来を完全に予測できる」というラプラスの悪魔への、定量的な反論です。
- $\lambda$ は「傾き（微分）の積の対数の平均」です。1次元では傾き、多次元ではヤコビ行列の積になります（次のノートブック）。

**次のノートブック**：[02_lorenz.ipynb](02_lorenz.ipynb) で、3次元の連続な系（ローレンツ系）を扱います。